# Comparison

# Theme

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import os, json, subprocess
from getpass import getpass

COLORS = {"bg":"#F5F5F5","teal":"#76ABAE","dark":"#303841","orange":"#FF5722","white":"#FFFFFF"}

# Setup

In [ ]:
os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")
print("Key loaded")

In [ ]:
from __future__ import annotations
from dataclasses import dataclass

@dataclass
class Finding:
    vuln_type: str
    severity: str
    line_number: int | None
    description: str
    source: str = "slither"

# Run Slither

In [ ]:
_DETECTOR_TO_CATEGORY = {
    "reentrancy-eth": "reentrancy", "reentrancy-no-eth": "reentrancy",
    "reentrancy-benign": "reentrancy", "reentrancy-events": "reentrancy",
    "unprotected-upgrade": "access_control", "arbitrary-send-eth": "access_control",
    "suicidal": "access_control", "tx-origin": "access_control",
    "timestamp": "time_manipulation",
    "low-level-calls": "unchecked_low_level_calls",
    "unchecked-transfer": "unchecked_low_level_calls",
    "unchecked-lowlevel": "unchecked_low_level_calls",
    "calls-loop": "denial_of_service", "costly-loop": "denial_of_service",
}
_IMPACT_TO_SEVERITY = {"High":"high","Medium":"medium","Low":"low","Informational":"low","Optimization":"low"}

def _parse_slither_json(raw_json):
    findings = []
    for detector in raw_json.get("results", {}).get("detectors", []):
        category = _DETECTOR_TO_CATEGORY.get(detector.get("check", ""), "unknown")
        severity = _IMPACT_TO_SEVERITY.get(detector.get("impact", ""), "low")
        description = detector.get("description", "").strip()
        line_number = None
        elements = detector.get("elements", [])
        if elements:
            lines = elements[0].get("source_mapping", {}).get("lines", [])
            if lines: line_number = lines[0]
        findings.append(Finding(category, severity, line_number, description))
    return findings

def run_slither(contract_path, timeout_s=30):
    try:
        result = subprocess.run(["slither", contract_path, "--json", "-"],
                                 capture_output=True, text=True, timeout=timeout_s)
    except subprocess.TimeoutExpired:
        return [Finding("analysis_error","high",None,f"Slither timed out on {contract_path}")]
    except FileNotFoundError:
        return [Finding("analysis_error","high",None,"slither not found")]
    if not result.stdout.strip():
        return [Finding("analysis_error","high",None,f"No JSON output: {result.stderr[:300]}")]
    try:
        raw_json = json.loads(result.stdout)
    except json.JSONDecodeError as e:
        return [Finding("analysis_error","high",None,f"JSON parse failed: {e}")]
    if not raw_json.get("success", True):
        return [Finding("analysis_error","high",None,f"Slither failed: {raw_json.get('error')}")]
    return _parse_slither_json(raw_json)

# Run Haiku

In [ ]:
import anthropic
client = anthropic.Anthropic()

JUDGE_PROMPT = """You are checking a Solidity contract for well-defined, pattern-matchable vulnerabilities ONLY. Do NOT reason about business logic intent.

Check specifically for:
- Reentrancy (external call before state update)
- Missing access control on sensitive functions
- Unbounded loops (potential DoS via gas exhaustion)
- Timestamp manipulation risk
- Unchecked low-level calls

Contract:
```solidity
{contract_source}
```

Respond ONLY with JSON: {{"findings": [{{"vuln_type": "<reentrancy|access_control|denial_of_service|time_manipulation|unchecked_low_level_calls>", "severity": "<high|medium|low>", "line_number": <int or null>, "description": "<one sentence>"}}]}}
If none found: {{"findings": []}}"""

def judge_contract(contract_source):
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=800, temperature=0,
        messages=[{"role":"user","content":JUDGE_PROMPT.format(contract_source=contract_source)}])
    text = resp.content[0].text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"): text = text[4:]
        text = text.strip()
    try:
        result = json.loads(text)
        return [Finding(f["vuln_type"], f.get("severity","low"), f.get("line_number"),
                         f.get("description",""), source="haiku_judge")
                for f in result.get("findings", [])]
    except (json.JSONDecodeError, KeyError) as e:
        return [Finding("analysis_error","high",None,f"Parse failed: {e}", source="haiku_judge")]

# Diagram

In [ ]:
MAIN_X, MAIN_W = 0.0, 6.0
DROP_X, DROP_W = 6.8, 5.2
GAP = 0.55
LINE = 0.34

def _h(detail):
    n = len(detail) if detail else 0
    return 0.62 + (0.30 + LINE * n if n else 0)

def draw_flowchart(title, stages):
    rows, y = [], 0.0
    for s in stages:
        if s["type"] == "drop":
            rows.append((s, None)); continue
        h = _h(s.get("detail"))
        rows.append((s, h)); y += h + GAP
    total = y + 1.0

    fig, ax = plt.subplots(figsize=(11, total * 0.78))
    fig.patch.set_facecolor(COLORS["bg"]); ax.set_facecolor(COLORS["bg"])
    ax.set_xlim(-0.4, 12.4); ax.set_ylim(0, total); ax.axis("off")

    cy = total - 0.7
    prev_bottom = None

    for s, h in rows:
        if s["type"] == "drop":
            d = s.get("detail") or []
            dh = 0.62 + 0.30 + LINE * len(d)
            top = prev_bottom - 0.10
            by = top - dh
            ax.add_patch(FancyBboxPatch((DROP_X, by), DROP_W, dh,
                boxstyle="round,pad=0.02,rounding_size=0.12",
                fc=COLORS["white"], ec=COLORS["orange"], lw=1.4))
            ax.text(DROP_X+0.28, top-0.36, s["label"], fontsize=10.5,
                    fontweight="bold", color=COLORS["orange"], va="center")
            ty = top - 0.36 - 0.42
            for ln in d:
                ax.text(DROP_X+0.28, ty, ln, fontsize=9, color=COLORS["dark"], va="center")
                ty -= LINE
            ymid = by + dh/2
            ax.annotate("", xy=(DROP_X-0.04, ymid), xytext=(MAIN_W+0.04, ymid),
                        arrowprops=dict(arrowstyle="-|>", color=COLORS["orange"],
                                        lw=1.4, shrinkA=0, shrinkB=0))
            continue

        final = s["type"] == "final"
        by = cy - h
        fc = COLORS["teal"] if final else COLORS["white"]
        ec = COLORS["teal"] if final else COLORS["dark"]
        tc = COLORS["white"] if final else COLORS["dark"]
        ax.add_patch(FancyBboxPatch((MAIN_X, by), MAIN_W, h,
            boxstyle="round,pad=0.02,rounding_size=0.12", fc=fc, ec=ec, lw=1.6))
        ax.text(MAIN_X+0.30, cy-0.36, s["label"], fontsize=11.5,
                fontweight="bold", color=tc, va="center")
        d = s.get("detail")
        if d:
            ty = cy - 0.36 - 0.42
            for ln in (d if isinstance(d, list) else [d]):
                ax.text(MAIN_X+0.30, ty, ln, fontsize=9, color=tc, va="center")
                ty -= LINE
        if prev_bottom is not None:
            ax.annotate("", xy=(MAIN_W/2, cy), xytext=(MAIN_W/2, prev_bottom),
                        arrowprops=dict(arrowstyle="-|>", color=COLORS["dark"],
                                        lw=1.5, shrinkA=0, shrinkB=0))
        prev_bottom = by
        cy = by - GAP

    ax.text(-0.4, total-0.18, title, fontsize=14, fontweight="bold",
            color=COLORS["dark"], va="center")
    plt.subplots_adjust(left=0.03, right=0.99, top=0.97, bottom=0.02)
    plt.show()